
# ECE 579 — Machine Learning for Engineering Applications
## 00. Data Preparation

This notebook prepares the photovoltaic power dataset for the five machine-learning problems.

### Objectives

- Load and validate the nine city–year datasets.
- Identify missing values and inconsistencies.
- Preserve chronological ordering and create reproducible 80/20 train–test splits.
- Prepare consistent datasets for classification, regression, dimensionality reduction, semi-supervised learning, and transfer learning.
- Save processed data and split information for use in the subsequent notebooks.

### Experimental Principles

- Chronological data splitting prevents temporal leakage.
- Missing target values are handled without introducing artificial ground-truth labels.
- Model-specific preprocessing is fitted only on permitted training data.
- Identical data partitions are reused across comparable experiments.

The processed datasets and split indices are stored in the `Data/Processed/` directory.


In [3]:

# ============================================================
# Cell 2 — Portable Environment and Project Setup
# ============================================================

import os
import sys
import platform
from pathlib import Path


# ------------------------------------------------------------
# 1. Detect Google Colab
# ------------------------------------------------------------

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")


# ------------------------------------------------------------
# 2. Locate the project directory
# ------------------------------------------------------------

PROJECT_NAME = "Code_base_v2"

# Optional override for custom locations.
# Example: PROJECT_ROOT_OVERRIDE = Path("/home/user/codebase_v2")
PROJECT_ROOT_OVERRIDE = None


def find_project_root():

    # Explicit configuration takes priority.
    configured = (
        PROJECT_ROOT_OVERRIDE
        or os.environ.get("ECE579_PROJECT_ROOT")
    )

    if configured:
        root = Path(configured).expanduser().resolve()

        if not root.is_dir():
            raise FileNotFoundError(
                f"Configured project directory not found: {root}"
            )

        return root

    # Local Jupyter: search the current folder and parents.
    current = Path.cwd().resolve()

    for location in [current, *current.parents]:

        if location.name.lower() == PROJECT_NAME.lower():
            return location

        candidate = location / PROJECT_NAME

        if candidate.is_dir():
            return candidate.resolve()

    # Colab: notebooks are stored in Google Drive,
    # but the runtime normally starts in /content.
    if IN_COLAB:

        drive_root = (
            Path("/content/drive/MyDrive")
            / "Colab Notebooks"
            / "ML_ECE_571_Project"
            / PROJECT_NAME
        )

        if drive_root.is_dir():
            return drive_root.resolve()

    raise FileNotFoundError(
        "Cannot locate codebase_v2.\n"
        "Open Jupyter from the project folder, or set "
        "PROJECT_ROOT_OVERRIDE to its location."
    )


PROJECT_ROOT = find_project_root()


# ------------------------------------------------------------
# 3. Define shared project directories
# ------------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "Data"
RAW_DIR = DATA_DIR / "Raw"
PROCESSED_DIR = DATA_DIR / "Processed"
SPLITS_DIR = PROCESSED_DIR / "Splits"

RESULTS_DIR = PROJECT_ROOT / "Results"
FIGURES_DIR = PROJECT_ROOT / "Figures"

PROBLEMS = ("P1", "P2", "P3", "P4", "P5")


# ------------------------------------------------------------
# 4. Create directories safely
# ------------------------------------------------------------

directories = [
    RAW_DIR,
    PROCESSED_DIR,
    SPLITS_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
]

directories += [
    RESULTS_DIR / problem
    for problem in PROBLEMS
]

directories += [
    FIGURES_DIR / problem
    for problem in PROBLEMS
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 5. Display environment information
# ------------------------------------------------------------

print("ECE 579 — PROJECT ENVIRONMENT")
print("=" * 65)

print(f"Platform       : {platform.system()}")
print(f"Architecture   : {platform.machine()}")
print(f"Python         : {platform.python_version()}")
print(f"Environment    : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

print(f"\nProject root   : {PROJECT_ROOT}")
print(f"Raw data       : {RAW_DIR.relative_to(PROJECT_ROOT)}")
print(f"Processed data : {PROCESSED_DIR.relative_to(PROJECT_ROOT)}")
print(f"Results        : {RESULTS_DIR.relative_to(PROJECT_ROOT)}")
print(f"Figures        : {FIGURES_DIR.relative_to(PROJECT_ROOT)}")

print("\nProblem directories:")
for problem in PROBLEMS:
    print(f"  {problem}: Results/{problem}/ | Figures/{problem}/")

print("\nProject directory setup completed successfully.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ECE 579 — PROJECT ENVIRONMENT
Platform       : Linux
Architecture   : x86_64
Python         : 3.13.16
Environment    : Google Colab

Project root   : /content/drive/MyDrive/Colab Notebooks/ML_ECE_571_Project/Code_base_v2
Raw data       : Data/Raw
Processed data : Data/Processed
Results        : Results
Figures        : Figures

Problem directories:
  P1: Results/P1/ | Figures/P1/
  P2: Results/P2/ | Figures/P2/
  P3: Results/P3/ | Figures/P3/
  P4: Results/P4/ | Figures/P4/
  P5: Results/P5/ | Figures/P5/

Project directory setup completed successfully.


In [4]:

# ============================================================
# Cell 3 — Dataset Discovery and Initial Validation
# ============================================================

import shutil
import pandas as pd
from openpyxl import load_workbook
from IPython.display import display


# ------------------------------------------------------------
# 1. Locate the original Excel workbook
# ------------------------------------------------------------

WORKBOOK_NAME = "Further Consolidated Data, HnL.xlsx"

# Prefer the portable Version 2 data directory.
WORKBOOK_PATH = RAW_DIR / WORKBOOK_NAME

# Search previous project locations only when necessary.
if not WORKBOOK_PATH.is_file():

    legacy_root = PROJECT_ROOT.parent / "Code_base"

    search_directories = [
        RAW_DIR,
        PROJECT_ROOT,
        PROJECT_ROOT.parent,
        legacy_root / "Data" / "Raw",
        legacy_root / "Data",
        legacy_root
    ]

    candidates = []

    for directory in search_directories:
        if directory.is_dir():
            candidates.extend(
                path
                for path in directory.glob("*.xlsx")
                if not path.name.startswith("~$")
            )

    # Remove duplicate paths.
    candidates = list(dict.fromkeys(candidates))

    matches = [
        path for path in candidates
        if path.name.casefold() == WORKBOOK_NAME.casefold()
    ]

    if len(matches) == 1:

        # Copy rather than move, preserving Version 1.
        shutil.copy2(matches[0], WORKBOOK_PATH)

        print(
            f"Copied workbook from:\n"
            f"  {matches[0]}\n"
            f"to:\n"
            f"  {WORKBOOK_PATH}\n"
        )

    elif len(matches) > 1:

        print("Multiple matching workbooks found:")
        for path in matches:
            print(" ", path)

        raise RuntimeError(
            "Place the intended workbook in Data/Raw "
            "to avoid selecting the wrong file."
        )

    else:

        print("Excel files found:")
        for path in candidates:
            print(" ", path.name, "—", path.parent)

        raise FileNotFoundError(
            f"Cannot find {WORKBOOK_NAME}.\n"
            f"Please copy the original Excel workbook into:\n"
            f"{RAW_DIR}"
        )


# ------------------------------------------------------------
# 2. Open workbook without modifying its contents
# ------------------------------------------------------------

workbook = load_workbook(
    WORKBOOK_PATH,
    read_only=True,
    data_only=True
)

sheet_names = workbook.sheetnames

print("ECE 579 — ORIGINAL DATASET INSPECTION")
print("=" * 75)

print(f"Workbook    : {WORKBOOK_PATH.name}")
print(f"Sheet count : {len(sheet_names)}")


# ------------------------------------------------------------
# 3. Inspect sheet dimensions and column names
# ------------------------------------------------------------

EXPECTED_COLUMNS = {
    "Year", "Month", "Day", "Hour", "Minute",
    "DHI", "DNI", "GHI",
    "Clearsky DHI", "Clearsky DNI", "Clearsky GHI",
    "Cloud Type", "Dew Point", "Solar Zenith Angle",
    "Surface Albedo", "Wind Speed",
    "Precipitable Water", "Wind Direction",
    "Relative Humidity", "Temperature",
    "Pressure", "Output Power"
}

sheet_summary = []
sheet_columns = {}

for sheet_name in sheet_names:

    sheet = workbook[sheet_name]

    header = next(
        sheet.iter_rows(
            min_row=1,
            max_row=1,
            values_only=True
        )
    )

    columns = [
        str(value).strip() if value is not None else ""
        for value in header
    ]

    sheet_columns[sheet_name] = columns

    missing_columns = sorted(
        EXPECTED_COLUMNS - set(columns)
    )

    sheet_summary.append({
        "Sheet": sheet_name,
        "Rows (approx.)": max(sheet.max_row - 1, 0),
        "Columns": len(columns),
        "Missing Expected Columns": (
            ", ".join(missing_columns)
            if missing_columns else "None"
        )
    })

workbook.close()

p5_dataset_summary = pd.DataFrame(sheet_summary)


# ------------------------------------------------------------
# 4. Display validation results
# ------------------------------------------------------------

print("\n1. WORKBOOK SHEETS")
display(p5_dataset_summary)

print("\n2. FIRST SHEET — ORIGINAL FEATURE COLUMNS")

for index, column in enumerate(
    sheet_columns[sheet_names[0]],
    start=1
):
    print(f"{index:2d}. {column}")


# ------------------------------------------------------------
# 5. Check the expected dataset structure
# ------------------------------------------------------------

all_columns_valid = all(
    EXPECTED_COLUMNS.issubset(set(columns))
    for columns in sheet_columns.values()
)

print("\n3. INITIAL VALIDATION")

print(f"Expected sheets (9)  : {len(sheet_names) == 9}")
print(f"Expected 22 columns  : {all(len(c) == 22 for c in sheet_columns.values())}")
print(f"Required columns     : {all_columns_valid}")

if len(sheet_names) != 9 or not all_columns_valid:
    print(
        "\nReview the workbook structure before "
        "proceeding with preprocessing."
    )
else:
    print(
        "\nWorkbook structure verified. "
        "Ready for detailed data-quality analysis."
    )

print(
    "\nNo source data, labels, or timestamps "
    "have been modified."
)


Copied workbook from:
  /content/drive/MyDrive/Colab Notebooks/ML_ECE_571_Project/Code_base/Data/Further Consolidated Data, HnL.xlsx
to:
  /content/drive/MyDrive/Colab Notebooks/ML_ECE_571_Project/Code_base_v2/Data/Raw/Further Consolidated Data, HnL.xlsx

ECE 579 — ORIGINAL DATASET INSPECTION
Workbook    : Further Consolidated Data, HnL.xlsx
Sheet count : 9

1. WORKBOOK SHEETS


,Sheet,Rows (approx.),Columns,Missing Expected Columns
0,Amhst 5hr-daily '18-'20,12056,23,None
1,Davis 5hr-daily '14-'16,12056,22,None
2,Huron 5hr-daily '14-'16,12056,22,None
3,Snt.Barb 5hr-daily '14-'16,12056,22,None
4,LaJolla 5hr-daily '14-'16,12056,22,None
5,Davis 5hr-daily '11-'16,24112,22,None
6,Huron 5hr-daily '11-'16,24112,22,None
7,Snt.Barb 5hr-daily '11-'16,24112,22,None
8,LaJolla 5hr-daily '11-'16,24112,22,None



2. FIRST SHEET — ORIGINAL FEATURE COLUMNS
 1. Year
 2. Month
 3. Day
 4. Hour
 5. Minute
 6. DHI
 7. DNI
 8. GHI
 9. Clearsky DHI
10. Clearsky DNI
11. Clearsky GHI
12. Cloud Type
13. Dew Point
14. Solar Zenith Angle
15. Surface Albedo
16. Wind Speed
17. Precipitable Water
18. Wind Direction
19. Relative Humidity
20. Temperature
21. Pressure
22. Output Power
23. 

3. INITIAL VALIDATION
Expected sheets (9)  : True
Expected 22 columns  : False
Required columns     : True

Workbook structure verified. Ready for detailed data-quality analysis.

No source data, labels, or timestamps have been modified.


In [5]:

# ============================================================
# Cell 4 — Data Quality Inspection
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

print("Loading all nine Excel sheets...")
RAW_SHEETS = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name=None,
    engine="openpyxl"
)

quality_records = []
missing_records = []

TIME_COLUMNS = ["Year", "Month", "Day", "Hour", "Minute"]

for sheet_name, df in RAW_SHEETS.items():

    df.columns = df.columns.astype(str).str.strip()

    # Identify unexpected columns and check their contents.
    extra_columns = [
        col for col in df.columns
        if col not in EXPECTED_COLUMNS
    ]

    nonempty_extra = {
        col: int(df[col].notna().sum())
        for col in extra_columns
        if df[col].notna().any()
    }

    # Identify missing or nonnumeric target values.
    power = pd.to_numeric(
        df["Output Power"],
        errors="coerce"
    )

    invalid_power = ~np.isfinite(
        power.to_numpy(dtype=float)
    )

    # Reconstruct timestamps for chronological checks.
    timestamps = pd.to_datetime(
        df[TIME_COLUMNS],
        errors="coerce"
    )

    quality_records.append({
        "Sheet": sheet_name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Power": int(invalid_power.sum()),
        "Invalid Timestamps": int(timestamps.isna().sum()),
        "Duplicate Timestamps": int(timestamps.duplicated().sum()),
        "Chronological": timestamps.is_monotonic_increasing,
        "Extra Columns": ", ".join(extra_columns) or "None",
        "Nonempty Extra": str(nonempty_extra) if nonempty_extra else "None"
    })

    # Record missing values across all required features.
    missing_counts = df[
        sorted(EXPECTED_COLUMNS)
    ].isna().sum()

    for col, count in missing_counts.items():
        if count > 0:
            missing_records.append({
                "Sheet": sheet_name,
                "Column": col,
                "Missing Values": int(count)
            })

    print(
        f"Checked: {sheet_name:30s} | "
        f"Missing power: {invalid_power.sum()}",
        flush=True
    )

    # Display the exact invalid-power observations.
    if invalid_power.any():
        print("  Observations with invalid Output Power:")
        display(
            df.loc[
                invalid_power,
                TIME_COLUMNS + ["Output Power"]
            ]
        )

quality_summary = pd.DataFrame(quality_records)
missing_summary = pd.DataFrame(missing_records)

print("\n1. DATASET QUALITY SUMMARY")
display(quality_summary)

print("\n2. MISSING VALUES BY COLUMN")

if missing_summary.empty:
    print("No missing values found.")
else:
    display(missing_summary)

print("\nData quality inspection complete.")
print("No original data have been modified.")


Loading all nine Excel sheets...
Checked: Amhst 5hr-daily '18-'20        | Missing power: 4
  Observations with invalid Output Power:


,Year,Month,Day,Hour,Minute,Output Power
10087,2020,7,6,10,0,NaN
10088,2020,7,6,10,30,NaN
10089,2020,7,6,11,0,NaN
10090,2020,7,6,11,30,NaN


Checked: Davis 5hr-daily '14-'16        | Missing power: 0
Checked: Huron 5hr-daily '14-'16        | Missing power: 0
Checked: Snt.Barb 5hr-daily '14-'16     | Missing power: 0
Checked: LaJolla 5hr-daily '14-'16      | Missing power: 0
Checked: Davis 5hr-daily '11-'16        | Missing power: 0
Checked: Huron 5hr-daily '11-'16        | Missing power: 0
Checked: Snt.Barb 5hr-daily '11-'16     | Missing power: 0
Checked: LaJolla 5hr-daily '11-'16      | Missing power: 0

1. DATASET QUALITY SUMMARY


,Sheet,Rows,Columns,Missing Power,Invalid Timestamps,Duplicate Timestamps,Chronological,Extra Columns,Nonempty Extra
0,Amhst 5hr-daily '18-'20,12056,22,4,0,0,True,None,None
1,Davis 5hr-daily '14-'16,12056,22,0,0,0,True,None,None
2,Huron 5hr-daily '14-'16,12056,22,0,0,0,True,None,None
3,Snt.Barb 5hr-daily '14-'16,12056,22,0,0,0,True,None,None
4,LaJolla 5hr-daily '14-'16,12056,22,0,0,0,True,None,None
5,Davis 5hr-daily '11-'16,24112,22,0,0,0,True,None,None
6,Huron 5hr-daily '11-'16,24112,22,0,0,0,True,None,None
7,Snt.Barb 5hr-daily '11-'16,24112,22,0,0,0,True,None,None
8,LaJolla 5hr-daily '11-'16,24112,22,0,0,0,True,None,None



2. MISSING VALUES BY COLUMN


,Sheet,Column,Missing Values
0,Amhst 5hr-daily '18-'20,Output Power,4



Data quality inspection complete.
No original data have been modified.


In [6]:

# ============================================================
# Cell 5 — Process Datasets and Save Chronological Splits
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Define the nine datasets and their output filenames
# ------------------------------------------------------------

SHEET_CONFIG = {
    "Amhst 5hr-daily '18-'20":    "Amherst_2018_2020",
    "Davis 5hr-daily '14-'16":    "Davis_2014_2016",
    "Huron 5hr-daily '14-'16":    "Huron_2014_2016",
    "Snt.Barb 5hr-daily '14-'16": "SantaBarbara_2014_2016",
    "LaJolla 5hr-daily '14-'16":  "LaJolla_2014_2016",
    "Davis 5hr-daily '11-'16":    "Davis_2011_2016",
    "Huron 5hr-daily '11-'16":    "Huron_2011_2016",
    "Snt.Barb 5hr-daily '11-'16": "SantaBarbara_2011_2016",
    "LaJolla 5hr-daily '11-'16":  "LaJolla_2011_2016"
}

TIME_COLUMNS = ["Year", "Month", "Day", "Hour", "Minute"]
TARGET_COLUMN = "Output Power"

# Explicit and consistent column ordering across datasets.
DATA_COLUMNS = [
    "Year", "Month", "Day", "Hour", "Minute",
    "DHI", "DNI", "GHI",
    "Clearsky DHI", "Clearsky DNI", "Clearsky GHI",
    "Cloud Type", "Dew Point", "Solar Zenith Angle",
    "Surface Albedo", "Wind Speed",
    "Precipitable Water", "Wind Direction",
    "Relative Humidity", "Temperature",
    "Pressure", "Output Power"
]

FEATURE_COLUMNS = [
    column for column in DATA_COLUMNS
    if column != TARGET_COLUMN
]

TRAIN_FRACTION = 0.80

assert len(SHEET_CONFIG) == 9
assert len(DATA_COLUMNS) == 22
assert set(SHEET_CONFIG) == set(RAW_SHEETS)


# ------------------------------------------------------------
# 2. Process one dataset
# ------------------------------------------------------------

def process_city_dataset(sheet_name, dataset_name):
    """
    Standardize column names, validate numeric data,
    construct timestamps, and create a chronological
    80/20 train-test split.

    Missing Output Power values are preserved in the
    processed data and marked in the split metadata.
    """

    df = RAW_SHEETS[sheet_name].copy()

    df.columns = df.columns.astype(str).str.strip()

    # --------------------------------------------------------
    # Validate and order columns
    # --------------------------------------------------------

    missing_columns = set(DATA_COLUMNS) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"{sheet_name}: Missing columns: {missing_columns}"
        )

    extra_columns = [
        column for column in df.columns
        if column not in DATA_COLUMNS
    ]

    # Only completely empty extra columns may be ignored.
    for column in extra_columns:
        if df[column].notna().any():
            raise ValueError(
                f"{sheet_name}: Unexpected nonempty "
                f"column: {column}"
            )

    df = df[DATA_COLUMNS].copy()

    # --------------------------------------------------------
    # Enforce numeric values
    # --------------------------------------------------------

    for column in DATA_COLUMNS:
        df[column] = pd.to_numeric(
            df[column],
            errors="raise"
        )

    # All predictor values must be finite.
    feature_values = df[
        FEATURE_COLUMNS
    ].to_numpy(dtype=float)

    if not np.isfinite(feature_values).all():
        raise ValueError(
            f"{sheet_name}: Invalid predictor values found."
        )

    # --------------------------------------------------------
    # Construct timestamps and preserve original row IDs
    # --------------------------------------------------------

    timestamps = pd.to_datetime(
        df[TIME_COLUMNS],
        errors="raise"
    )

    if not timestamps.is_monotonic_increasing:
        raise ValueError(
            f"{sheet_name}: Timestamps are not chronological."
        )

    if timestamps.duplicated().any():
        raise ValueError(
            f"{sheet_name}: Duplicate timestamps detected."
        )

    df.insert(0, "Timestamp", timestamps)
    df.insert(0, "row_index", np.arange(len(df)))

    # row_index and Timestamp are metadata, not predictors.

    # --------------------------------------------------------
    # Identify valid output-power labels
    # --------------------------------------------------------

    valid_power = np.isfinite(
        df[TARGET_COLUMN].to_numpy(dtype=float)
    )

    # --------------------------------------------------------
    # Create chronological 80/20 split
    # --------------------------------------------------------

    n_samples = len(df)

    n_train = int(np.floor(
        TRAIN_FRACTION * n_samples
    ))

    if not 0 < n_train < n_samples:
        raise ValueError(
            f"{sheet_name}: Invalid split size."
        )

    split_labels = np.where(
        np.arange(n_samples) < n_train,
        "train",
        "test"
    )

    split_df = pd.DataFrame({
        "row_index": df["row_index"],
        "Timestamp": df["Timestamp"],
        "split": split_labels,
        "valid_output_power": valid_power
    })

    # --------------------------------------------------------
    # Verify split integrity
    # --------------------------------------------------------

    train_mask = split_labels == "train"
    test_mask = split_labels == "test"

    assert train_mask.sum() == n_train
    assert test_mask.sum() == n_samples - n_train

    assert (
        df.loc[train_mask, "Timestamp"].max()
        < df.loc[test_mask, "Timestamp"].min()
    )

    # Missing training targets would require additional
    # handling before training supervised regressors.
    if not valid_power[train_mask].all():
        raise ValueError(
            f"{sheet_name}: Missing training target detected."
        )

    missing_test_targets = int(
        (~valid_power[test_mask]).sum()
    )

    # --------------------------------------------------------
    # Save the processed dataset and its split manifest
    # --------------------------------------------------------

    data_path = PROCESSED_DIR / f"{dataset_name}.csv"

    split_path = (
        SPLITS_DIR
        / f"{dataset_name}_chronological_80_20.csv"
    )

    df.to_csv(data_path, index=False)
    split_df.to_csv(split_path, index=False)

    return {
        "Dataset": dataset_name,
        "Total Rows": n_samples,
        "Train": int(train_mask.sum()),
        "Test": int(test_mask.sum()),
        "Valid Test": int(valid_power[test_mask].sum()),
        "Missing Test Power": missing_test_targets,
        "Start": df["Timestamp"].min().date(),
        "End": df["Timestamp"].max().date()
    }


# ------------------------------------------------------------
# 3. Process all nine datasets
# ------------------------------------------------------------

processing_records = []

print("ECE 579 — PROCESSING PHOTOVOLTAIC DATASETS")
print("=" * 90)

for sheet_name, dataset_name in SHEET_CONFIG.items():

    record = process_city_dataset(
        sheet_name,
        dataset_name
    )

    processing_records.append(record)

    print(
        f"{dataset_name:28s} | "
        f"Train: {record['Train']:5d} | "
        f"Test: {record['Test']:5d} | "
        f"Missing test power: "
        f"{record['Missing Test Power']}",
        flush=True
    )


# ------------------------------------------------------------
# 4. Display final dataset summary
# ------------------------------------------------------------

processed_summary = pd.DataFrame(
    processing_records
)

assert len(processed_summary) == 9

print("\n1. PROCESSED DATASETS AND CHRONOLOGICAL SPLITS")
display(processed_summary)

print("\n2. DATA STORAGE")

print(f"Processed CSV files : {PROCESSED_DIR}")
print(f"Split manifests     : {SPLITS_DIR}")

print("\n3. DATA INTEGRITY")

print(
    "All datasets preserve original observation order."
)
print(
    "All splits use the first 80% for training "
    "and the final 20% for testing."
)
print(
    "Missing output-power labels remain explicitly marked."
)
print(
    "No model scalers, encoders, or classification "
    "thresholds have been fitted."
)

print("\nAll nine datasets processed successfully.")


ECE 579 — PROCESSING PHOTOVOLTAIC DATASETS
Amherst_2018_2020            | Train:  9644 | Test:  2412 | Missing test power: 4
Davis_2014_2016              | Train:  9644 | Test:  2412 | Missing test power: 0
Huron_2014_2016              | Train:  9644 | Test:  2412 | Missing test power: 0
SantaBarbara_2014_2016       | Train:  9644 | Test:  2412 | Missing test power: 0
LaJolla_2014_2016            | Train:  9644 | Test:  2412 | Missing test power: 0
Davis_2011_2016              | Train: 19289 | Test:  4823 | Missing test power: 0
Huron_2011_2016              | Train: 19289 | Test:  4823 | Missing test power: 0
SantaBarbara_2011_2016       | Train: 19289 | Test:  4823 | Missing test power: 0
LaJolla_2011_2016            | Train: 19289 | Test:  4823 | Missing test power: 0

1. PROCESSED DATASETS AND CHRONOLOGICAL SPLITS


,Dataset,Total Rows,Train,Test,Valid Test,Missing Test Power,Start,End
0,Amherst_2018_2020,12056,9644,2412,2408,4,2018-01-01,2020-12-31
1,Davis_2014_2016,12056,9644,2412,2412,0,2014-01-01,2016-12-31
2,Huron_2014_2016,12056,9644,2412,2412,0,2014-01-01,2016-12-31
3,SantaBarbara_2014_2016,12056,9644,2412,2412,0,2014-01-01,2016-12-31
4,LaJolla_2014_2016,12056,9644,2412,2412,0,2014-01-01,2016-12-31
5,Davis_2011_2016,24112,19289,4823,4823,0,2011-01-01,2016-12-31
6,Huron_2011_2016,24112,19289,4823,4823,0,2011-01-01,2016-12-31
7,SantaBarbara_2011_2016,24112,19289,4823,4823,0,2011-01-01,2016-12-31
8,LaJolla_2011_2016,24112,19289,4823,4823,0,2011-01-01,2016-12-31



2. DATA STORAGE
Processed CSV files : /content/drive/MyDrive/Colab Notebooks/ML_ECE_571_Project/Code_base_v2/Data/Processed
Split manifests     : /content/drive/MyDrive/Colab Notebooks/ML_ECE_571_Project/Code_base_v2/Data/Processed/Splits

3. DATA INTEGRITY
All datasets preserve original observation order.
All splits use the first 80% for training and the final 20% for testing.
Missing output-power labels remain explicitly marked.
No model scalers, encoders, or classification thresholds have been fitted.

All nine datasets processed successfully.


In [7]:

# ============================================================
# Cell 6 — Verify Saved Datasets and Chronological Splits
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display


def verify_saved_dataset(dataset_name):
    """
    Independently reload and validate a processed dataset
    and its chronological train/test split manifest.
    """

    data_path = PROCESSED_DIR / f"{dataset_name}.csv"

    split_path = (
        SPLITS_DIR /
        f"{dataset_name}_chronological_80_20.csv"
    )

    assert data_path.is_file(), f"Missing: {data_path}"
    assert split_path.is_file(), f"Missing: {split_path}"

    # --------------------------------------------------------
    # Reload saved files
    # --------------------------------------------------------

    df = pd.read_csv(
        data_path,
        parse_dates=["Timestamp"]
    )

    splits = pd.read_csv(
        split_path,
        parse_dates=["Timestamp"]
    )

    # --------------------------------------------------------
    # Verify schema and row alignment
    # --------------------------------------------------------

    assert len(df) == len(splits)

    assert list(df.columns) == (
        ["row_index", "Timestamp"] + DATA_COLUMNS
    )

    assert set(splits.columns) == {
        "row_index",
        "Timestamp",
        "split",
        "valid_output_power"
    }

    np.testing.assert_array_equal(
        df["row_index"].to_numpy(),
        np.arange(len(df))
    )

    np.testing.assert_array_equal(
        df["row_index"].to_numpy(),
        splits["row_index"].to_numpy()
    )

    np.testing.assert_array_equal(
        df["Timestamp"].to_numpy(),
        splits["Timestamp"].to_numpy()
    )

    assert df["Timestamp"].is_monotonic_increasing
    assert not df["Timestamp"].duplicated().any()

    # --------------------------------------------------------
    # Verify chronological 80/20 split
    # --------------------------------------------------------

    n_train = int(np.floor(
        TRAIN_FRACTION * len(df)
    ))

    expected_splits = np.where(
        np.arange(len(df)) < n_train,
        "train",
        "test"
    )

    np.testing.assert_array_equal(
        splits["split"].to_numpy(),
        expected_splits
    )

    train_mask = splits["split"] == "train"
    test_mask = splits["split"] == "test"

    assert (
        df.loc[train_mask, "Timestamp"].max()
        < df.loc[test_mask, "Timestamp"].min()
    )

    # --------------------------------------------------------
    # Verify features and target validity
    # --------------------------------------------------------

    predictor_values = df[
        FEATURE_COLUMNS
    ].to_numpy(dtype=float)

    assert np.isfinite(predictor_values).all()

    power = df[TARGET_COLUMN].to_numpy(dtype=float)
    valid_power = np.isfinite(power)

    np.testing.assert_array_equal(
        splits["valid_output_power"].to_numpy(
            dtype=bool
        ),
        valid_power
    )

    # No missing regression targets in training.
    assert valid_power[train_mask.to_numpy()].all()

    valid_test = int(
        valid_power[test_mask.to_numpy()].sum()
    )

    missing_test = int(test_mask.sum()) - valid_test

    return {
        "Dataset": dataset_name,
        "Rows": len(df),
        "Train": int(train_mask.sum()),
        "Test": int(test_mask.sum()),
        "Valid Test Power": valid_test,
        "Missing Test Power": missing_test,
        "Split Verified": True
    }


# ------------------------------------------------------------
# 1. Verify all nine saved datasets
# ------------------------------------------------------------

verification_records = []

print("ECE 579 — SAVED DATASET VERIFICATION")
print("=" * 85)

for dataset_name in SHEET_CONFIG.values():

    record = verify_saved_dataset(dataset_name)

    verification_records.append(record)

    print(
        f"[PASS] {dataset_name}",
        flush=True
    )


# ------------------------------------------------------------
# 2. Display the verification summary
# ------------------------------------------------------------

verification_summary = pd.DataFrame(
    verification_records
)

assert len(verification_summary) == 9
assert verification_summary["Split Verified"].all()

# Check the expected missing Amherst labels.
amherst_record = verification_summary.loc[
    verification_summary["Dataset"]
    == "Amherst_2018_2020"
].iloc[0]

assert amherst_record["Missing Test Power"] == 4
assert amherst_record["Valid Test Power"] == 2408

# All other datasets have complete output-power targets.
other_cities = verification_summary.loc[
    verification_summary["Dataset"]
    != "Amherst_2018_2020"
]

assert (
    other_cities["Missing Test Power"] == 0
).all()

print("\nVERIFIED DATASETS")
display(verification_summary)

print("\nAll nine saved datasets passed verification.")
print("Chronological splits and row alignment confirmed.")
print("Data preparation is complete.")


ECE 579 — SAVED DATASET VERIFICATION
[PASS] Amherst_2018_2020
[PASS] Davis_2014_2016
[PASS] Huron_2014_2016
[PASS] SantaBarbara_2014_2016
[PASS] LaJolla_2014_2016
[PASS] Davis_2011_2016
[PASS] Huron_2011_2016
[PASS] SantaBarbara_2011_2016
[PASS] LaJolla_2011_2016

VERIFIED DATASETS


,Dataset,Rows,Train,Test,Valid Test Power,Missing Test Power,Split Verified
0,Amherst_2018_2020,12056,9644,2412,2408,4,True
1,Davis_2014_2016,12056,9644,2412,2412,0,True
2,Huron_2014_2016,12056,9644,2412,2412,0,True
3,SantaBarbara_2014_2016,12056,9644,2412,2412,0,True
4,LaJolla_2014_2016,12056,9644,2412,2412,0,True
5,Davis_2011_2016,24112,19289,4823,4823,0,True
6,Huron_2011_2016,24112,19289,4823,4823,0,True
7,SantaBarbara_2011_2016,24112,19289,4823,4823,0,True
8,LaJolla_2011_2016,24112,19289,4823,4823,0,True



All nine saved datasets passed verification.
Chronological splits and row alignment confirmed.
Data preparation is complete.
